# Week 3 · Notebook 03: Fit a Transit, Measure a Planet
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nightwings24/exoplanet-transit-winter-project/blob/main/notebooks/03_fit_transit_radius.ipynb)

**Goal:** go from "there is a dip" to **"this planet is 1.5 ± 0.02 Earth radii"**: fit a model, get honest
error bars, and compare with the published value. **Time:** 5–7 hours.

### 🎯 By the end of this notebook you can
1. explain why the first-pass detrending biases the depth, and fix it;
2. explain what "fitting" means using χ², and get an error bar from Δχ² = 1 by hand;
3. fit a trapezoid model, check its residuals, and bootstrap the error bar;
4. propagate errors from depth and star radius into planet radius;
5. explain limb darkening and the impact-parameter degeneracy, and use a stellar-density prior to break it;
6. compare your result with the literature in units of σ.

### 🧭 Roadmap
| § | Topic |
|---|---|
| 1 | The 10-second estimate |
| 2 | Re-detrend with transits masked |
| 3 | What is fitting? χ² by hand |
| 4 | The trapezoid model and fit |
| 5 | Residuals, error propagation, bootstrap |
| 6 | The physical model: limb darkening, degeneracy, density prior |
| 7 | Comparison with the literature |
| 8 | Your target |

**Reading:**
- Winn 2010, *Transits and Occultations* ([arXiv:1001.2010](https://arxiv.org/abs/1001.2010)), Sec. 2 (the geometry)
- Seager & Mallén-Ornelas 2003 ([arXiv:astro-ph/0206228](https://arxiv.org/abs/astro-ph/0206228)): what a transit's shape tells you
- Kreidberg 2015, *batman* ([arXiv:1507.08285](https://arxiv.org/abs/1507.08285)) and its [docs](https://lkreidberg.github.io/batman/docs/html/index.html)
- Hogg, Bovy & Lang 2010, *Data analysis recipes: fitting a model to data* ([arXiv:1008.4686](https://arxiv.org/abs/1008.4686)), Sec. 1–3
<details><summary><b>📎 How to read this notebook (click to open)</b></summary>

| Symbol | Meaning |
|---|---|
| 🔮 **Predict** | Before running the next cell, write down what you expect. Being wrong is how you learn. |
| 👀 **What to look for** | How to read the output or plot you just made. |
| ✏️ **Your turn** | An exercise. 🟢 everyone · 🟡 stretch · 🔴 challenge. |
| 💡 / ✅ | Click to reveal a hint or a worked solution. Try first, then peek. |
| ⚠️ **Pitfall** | A mistake that real researchers make. |
| 🧠 **Concept check** | Short quiz. Answers are hidden under each question. |
| `check(...)` | Self-check: prints ✅ when your exercise answer is right. |

Run cells **top to bottom** with **Shift+Enter**. If things get weird: *Runtime → Restart and run all* (Colab) or *Kernel → Restart & Run All* (Jupyter).
</details>

In [ ]:
# ⚙️ SETUP: run this cell first, every time you open the notebook. Works in Google Colab and locally.
import os, sys
REPO_URL = "https://github.com/Nightwings24/exoplanet-transit-winter-project.git"  # <- change to YOUR repo
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("exoplanet-transit-winter-project"):
        !git clone -q {REPO_URL}
    %cd exoplanet-transit-winter-project
    !pip install -q lightkurve batman-package wotan tqdm
# Make `import transitpipe` work without installing the package
ROOT = os.path.abspath("." if os.path.exists("src") else "..")
sys.path.insert(0, os.path.join(ROOT, "src"))
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (10, 3.5)})

def check(name, value, expected, rtol=0.02):
    """Self-check for the exercises. Prints ✅, ❌, or ⏳ (not attempted yet)."""
    if value is None:
        print(f"⏳ {name}: not attempted yet (replace None with your answer)")
        return
    try:
        ok = bool(np.allclose(value, expected, rtol=rtol))
    except Exception:
        ok = value == expected
    print(f"✅ {name}: correct!" if ok else f"❌ {name}: not quite (you gave {value!r}). Check the hint.")

print("ready, repo root:", ROOT)

In [ ]:
from transitpipe import io, detrend, search, fit, plotting, utils

CACHE = os.path.join(ROOT, "data", "kepler10_q3q4.csv")
if os.path.exists(CACHE):
    lc = io.from_csv(CACHE)
else:
    lc = io.get_lightcurve("Kepler-10", mission="Kepler", exptime="long", quarter=[3, 4])
    os.makedirs(os.path.dirname(CACHE), exist_ok=True)
    io.to_csv(lc, CACHE)
STAR = {"rstar": 1.065, "rstar_err": 0.009, "mstar": 0.910, "mstar_err": 0.021}
PUBLISHED = {"rp": 1.47, "rp_err": 0.03}            # Kepler-10 b, Dumusque et al. 2014
lc.meta.update(STAR)

# Week 2 in three lines: detrend -> clip -> search
flat = detrend.sigma_clip(detrend.flatten(lc, method="median", window_days=0.5))
res = search.bls(flat, max_period=10)
print(res.summary())

## 1. The 10-second estimate
$R_p = \sqrt{\delta}\,R_\star$ using the BLS box depth. Write it down, then see how much better we can do.

In [ ]:
rp_quick = utils.depth_to_rearth(res.depth, STAR["rstar"])
print(f"Quick estimate from the BLS depth: Rp ≈ {rp_quick:.2f} R_earth   (published: {PUBLISHED['rp']})")

## 2. Re-detrend with the transits hidden (do NOT skip this)
In pass 1 the smoother *saw* the transits. Inside any window where the star is brightening or fading, the
few low in-transit points shift the median down, so the trend dips under each transit and the transit looks
**shallower** than it is. Now that we know *where* the transits are, we hide them from the smoother and detrend
the raw light curve again. Real pipelines (Kepler's included) do exactly this.

🔮 **Predict:** will the depth go up or down in pass 2? By a lot or a little?

In [ ]:
flat2, trend2 = detrend.flatten_masked(lc, res.period, res.t0, res.duration, window_days=0.5, return_trend=True)
flat2 = detrend.sigma_clip(flat2)
_, trend1 = detrend.flatten(lc, method="median", window_days=0.5, return_trend=True)

for name, l in [("pass 1", flat), ("pass 2", flat2)]:
    core = detrend.transit_mask(l.time, res.period, res.t0, res.duration, pad=0.5)  # use each curve's own time
    print(f"mean in-transit depth, {name}: {(1 - l.flux[core].mean()) * 1e6:.0f} ppm")

# How different are the two trends? Fold their difference on the planet's period and average.
hours = utils.phase_fold(lc.time, res.period, res.t0) * res.period * 24
diff_ppm = (trend1 - trend2) * 1e6
xb, yb = utils.bin_series(hours, diff_ppm, bins=40)
plt.figure(figsize=(7, 3.5))
plt.plot(xb, yb, "o-"); plt.axhline(0, color="0.5", ls=":")
plt.xlabel("Hours from mid-transit"); plt.ylabel("trend pass 1 − pass 2 [ppm]")
plt.title("The pass-1 trend sits low"); plt.show()

👀 The pass-1 trend sits ~11 ppm low wherever its 0.5-day window overlaps a transit: within about ±6 hours of
each one (half a window, plus the transit itself). Kepler-10 b's orbit is only 20 hours, so that is most of the
orbit, including the whole transit. Dividing by a trend that is too low lifts the data, so the transit looks
~7 % too shallow (174 vs 186 ppm above). On an active star the bias was ~20 % (Notebook 02's window table).
**Always fit on the pass-2 light curve.**

## 3. What does "fitting" mean? χ² by hand
A **model** predicts the flux at each time from a few **parameters**. A **fit** finds the parameters that make
the model closest to the data. "Closest" is measured by **chi-squared**:

$$\chi^2 = \sum_i \left(\frac{f_i - m(t_i)}{\sigma_i}\right)^2$$

(the sum of squared residuals, each in units of its error bar). The best fit **minimises** χ². The error bar on a
parameter is how far you can move it before χ² rises by **1** above its minimum (for one parameter, Gaussian
noise). Let's see this with the simplest model: a box of fixed position and width, where only the **depth** is free.

In [ ]:
t, f = flat2.time, flat2.flux
sigma = detrend.estimate_noise_ppm(flat2) * 1e-6             # noise per point
box = detrend.transit_mask(t, res.period, res.t0, res.duration, pad=1.0)   # True inside the box

depth_grid = np.linspace(0, 300e-6, 601)
chi2 = np.array([np.sum(((f - (1 - d * box)) / sigma) ** 2) for d in depth_grid])

best = depth_grid[chi2.argmin()]
inside = depth_grid[chi2 <= chi2.min() + 1]                  # all depths within Δχ² = 1
err = (inside.max() - inside.min()) / 2
print(f"best depth = {best * 1e6:.1f} ± {err * 1e6:.1f} ppm  (Δχ² = 1)")
print(f"analytic   σ/√N_in = {sigma / np.sqrt(box.sum()) * 1e6:.1f} ppm  ({box.sum()} in-box points)")

plt.figure(figsize=(6, 3.5))
plt.plot(depth_grid * 1e6, chi2 - chi2.min())
plt.axhline(1, color="C3", ls="--", label="Δχ² = 1"); plt.ylim(0, 10); plt.xlim((best - 5 * err) * 1e6, (best + 5 * err) * 1e6)
plt.xlabel("box depth [ppm]"); plt.ylabel("χ² − χ²_min"); plt.legend(); plt.show()

👀 χ² is a parabola; its bottom is the best depth and its width at Δχ² = 1 is the error bar, which matches
the √N formula from Notebook 00. With 4 free parameters we can't scan a grid by hand any more (100⁴ models!),
so we let an optimiser (`scipy.optimize.least_squares`) walk downhill, and get errors from the curvature of χ²
at the bottom (the *covariance matrix*). Same idea, more dimensions.

## 4. The trapezoid model
A box has vertical walls, but a real planet takes time to cross the edge of the star: the **ingress** and
**egress**. A trapezoid captures that with 4 parameters:

In [ ]:
tt = np.linspace(-0.1, 0.1, 1000)
m = fit.trapezoid_model(tt, 0.0, 10.0, 1e-3, 0.08, 0.015)
plt.figure(figsize=(7, 3.5)); plt.plot(tt * 24, (m - 1) * 1e6, lw=2)
plt.annotate("", xy=(-0.04 * 24, 150), xytext=(0.04 * 24, 150), arrowprops=dict(arrowstyle="<->"))
plt.text(0, 200, r"$T_{14}$ (total duration)", ha="center")
plt.annotate("", xy=(-0.04 * 24, -1100), xytext=(-0.025 * 24, -1100), arrowprops=dict(arrowstyle="<->"))
plt.text(-0.0325 * 24, -1250, r"$\tau$ (ingress)", ha="center")
plt.annotate("", xy=(0.03 * 24, 0), xytext=(0.03 * 24, -1000), arrowprops=dict(arrowstyle="<->"))
plt.text(0.033 * 24, -500, r"$\delta$ (depth)")
plt.axvline(0, color="0.6", ls=":"); plt.text(0.1, -1350, r"$t_0$")
plt.ylim(-1400, 350); plt.xlabel("Hours from mid-transit"); plt.ylabel("Flux − 1 [ppm]")
plt.title("Trapezoid model: 4 parameters"); plt.show()

In [ ]:
ft = fit.fit_single_transit(flat2, res, model="trapezoid")
print(ft.summary())
model = lambda t: fit.trapezoid_model(t, ft.t0, ft.period, ft.depth, ft.duration, ft.extra["ingress"])
plotting.plot_fold(flat2, ft.period, ft.t0, model_func=model, window_hours=5); plt.show()

## 5. Is the fit any good? Residuals
**Residuals** = data − model. If the model is right, residuals look like pure noise: centred on zero,
no structure in time or phase, with the expected scatter. *Always* plot them.

In [ ]:
near = detrend.transit_mask(flat2.time, ft.period, ft.t0, ft.duration, pad=6)    # a window around each transit
resid = flat2.flux[near] - model(flat2.time[near])
hours = utils.phase_fold(flat2.time[near], ft.period, ft.t0) * ft.period * 24

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(hours, resid * 1e6, ".", ms=1, color="0.6")
xb, yb = utils.bin_series(hours, resid, bins=40); ax[0].plot(xb, yb * 1e6, "o", color="C0")
ax[0].axhline(0, color="C3"); ax[0].set_ylim(-400, 400)
ax[0].set_xlabel("Hours from mid-transit"); ax[0].set_ylabel("residual [ppm]"); ax[0].set_title("Residuals vs phase")
ax[1].hist(resid * 1e6, bins=60, density=True, alpha=0.7)
x = np.linspace(-500, 500, 200); s = resid.std() * 1e6
ax[1].plot(x, np.exp(-0.5 * (x / s) ** 2) / (s * np.sqrt(2 * np.pi)), "C3", label="Gaussian")
ax[1].set_xlabel("residual [ppm]"); ax[1].legend(); ax[1].set_title("Residual distribution")
plt.tight_layout(); plt.show()
print(f"residual rms {resid.std() * 1e6:.0f} ppm vs noise estimate {detrend.estimate_noise_ppm(flat2):.0f} ppm")

👀 The binned residuals sit on zero through ingress, transit and egress: the model captures the shape. A
systematic "W" or "M" pattern in the binned residuals would mean the wrong shape (e.g. limb darkening, §6).

### Error propagation
$R_p = k R_\star$ with $k = \sqrt{\delta}$. Fractional errors add in quadrature:
$$\left(\frac{\sigma_{R_p}}{R_p}\right)^2 = \left(\frac{\sigma_k}{k}\right)^2 + \left(\frac{\sigma_{R_\star}}{R_\star}\right)^2,\qquad \frac{\sigma_k}{k} = \frac{1}{2}\frac{\sigma_\delta}{\delta}$$
🔮 **Predict:** for Kepler-10, which term is bigger: the light curve or the star?

In [ ]:
frac_k = ft.rp_rs_err / ft.rp_rs
frac_star = STAR["rstar_err"] / STAR["rstar"]
print(f"fractional error from the light curve: {frac_k:.2%}")
print(f"fractional error from the star:        {frac_star:.2%}")
print(f"total: {np.hypot(frac_k, frac_star):.2%}  ->  Rp = {ft.rp_rearth:.2f} ± {ft.rp_rearth_err:.2f} R_earth")

👀 For a well-studied star like Kepler-10 (asteroseismology gives <1 % radius) the light curve dominates.
For a typical TESS star with a 5–7 % catalogue radius error, **the star** dominates: you can't know the planet
better than you know its star.

### Check the error bar with a bootstrap
The covariance error assumes well-behaved Gaussian noise. A **bootstrap** checks it with brute force: make many
fake data sets (best model + resampled residuals), refit each, and look at the spread. No calculus needed.

In [ ]:
depths = fit.bootstrap_depth(flat2, ft, n_boot=100)
print(f"covariance depth error: {ft.depth_err * 1e6:.1f} ppm   bootstrap: {depths.std() * 1e6:.1f} ppm")
plt.figure(figsize=(6, 3.5)); plt.hist(depths * 1e6, bins=25)
plt.axvline(ft.depth * 1e6, color="C3", label="best fit")
plt.xlabel("depth [ppm]"); plt.ylabel("count"); plt.title("100 bootstrap refits"); plt.legend(); plt.show()

## 6. The physical model: batman (Mandel & Agol 2002)
### Limb darkening
Stars are **brighter at the centre** of their disk than at the edge (we see hotter, deeper layers in the middle).
So a planet crossing the centre blocks *brighter* light, and the dip is **deeper than $(R_p/R_\star)^2$**, with
a rounded bottom instead of a flat one. A trapezoid can't know this; the physical model `batman` can.
Limb darkening is described by coefficients $u_1, u_2$ (quadratic law).

🔮 **Predict:** same planet, same orbit: rank the three curves below by depth.

In [ ]:
try:
    import batman  # noqa: F401
    HAVE_BATMAN = True
except Exception:
    HAVE_BATMAN = False
    print("batman is not installed: run `pip install batman-package setuptools` to do Section 6.")

if HAVE_BATMAN:
    tt = np.linspace(-0.06, 0.06, 600)
    k = 0.0127
    plt.figure(figsize=(7, 3.5))
    for u, lab in [((0.0, 0.0), "no limb darkening"), ((0.40, 0.26), "Sun-like (Kepler band)"),
                   ((0.80, 0.0), "strong limb darkening")]:
        mm = fit.batman_model(tt, 0.0, 0.8375, k, 3.4, 0.3, u)
        plt.plot(tt * 24, (mm - 1) * 1e6, label=f"{lab}: depth {(1 - mm.min()) * 1e6:.0f} ppm")
    plt.axhline(-k ** 2 * 1e6, color="0.5", ls=":", label=f"(Rp/Rs)² = {k ** 2 * 1e6:.0f} ppm")
    plt.xlabel("Hours from mid-transit"); plt.ylabel("Flux − 1 [ppm]"); plt.legend(fontsize=8); plt.show()

👀 Same planet, different depths. Fitting a limb-darkened transit with a flat-bottomed trapezoid (or ignoring
limb darkening) biases $R_p/R_\star$. Heller (2019) quantified this for Kepler planets.

### The impact-parameter degeneracy
The **impact parameter** $b$ is how far from the star's centre the planet crosses (0 = through the middle,
1 = grazing the edge). At modest SNR, a *small planet crossing the centre* and a *bigger planet crossing near
the edge*, where the star is dimmer and the chord is shorter, produce almost the same light curve. Let's prove it: fit twice with $b$ held fixed at 0 and at 0.8.

In [ ]:
if HAVE_BATMAN:
    EXPTIME = 0.0204                     # Kepler long cadence = 29.4 min
    for b_fixed in [0.0, 0.8]:
        fb_ = fit.fit_single_transit(flat2, res, model="batman", fix_b=b_fixed, u=(0.40, 0.26), exptime_days=EXPTIME)
        print(f"b fixed at {b_fixed}: Rp = {fb_.rp_rearth:.2f} R_earth, residual rms = {fb_.residual_rms_ppm:.2f} ppm, "
              f"implied stellar density = {fb_.extra['rho_star_cgs']:.2f} g/cm³")
    rho, rho_err = utils.stellar_density(STAR["rstar"], STAR["mstar"], STAR["rstar_err"], STAR["mstar_err"])
    print(f"\ncatalogue stellar density: {rho:.2f} ± {rho_err:.2f} g/cm³  (Sun = 1.41)")

👀 The two fits are almost equally good (nearly identical residual rms) but give **different radii**. The light
curve alone can't decide. The tie-breaker is the **implied stellar density**: for a given period, the transit
duration fixes $a/R_\star$, and Kepler's third law turns that into the star's density,
$\rho_\star = \dfrac{3\pi}{GP^2}\left(\dfrac{a}{R_\star}\right)^3$. Only one of the two fits implies a density
matching the catalogue star. So we add the catalogue density as a **prior**: an extra term in χ² that penalises
fits whose implied density disagrees with it. This is exactly what published papers do.

In [ ]:
if HAVE_BATMAN:
    fb = fit.fit_single_transit(flat2, res, model="batman", rho_star=rho, rho_star_err=rho_err,
                                u=(0.40, 0.26), exptime_days=EXPTIME)
    print(fb.summary())
    e = fb.extra
    print(f"a/Rs = {e['a_rs']:.2f} ± {e['a_rs_err']:.2f},  b = {e['b']:.2f} ± {e['b_err']:.2f}")
    bm = lambda t: fit.batman_model(t, fb.t0, fb.period, fb.rp_rs, e["a_rs"], e["b"], e["u"], EXPTIME)
    plotting.plot_fold(flat2, fb.period, fb.t0, model_func=bm, window_hours=5, title="batman fit + density prior")
    plt.show()

### Why `exptime_days` matters
Each Kepler long-cadence point is a **30-minute average**, comparable to the ~10-minute ingress. A model
evaluated at the mid-exposure instant is too sharp. `batman` *supersamples*: it computes the model at several
moments inside each exposure and averages them. 🔮 **Predict**, then run: what changes without it?

In [ ]:
if HAVE_BATMAN:
    fb_ns = fit.fit_single_transit(flat2, res, model="batman", rho_star=rho, rho_star_err=rho_err, u=(0.40, 0.26))
    print(f"with supersampling:    Rp/Rs = {fb.rp_rs:.4f}, b = {fb.extra['b']:.2f}")
    print(f"without supersampling: Rp/Rs = {fb_ns.rp_rs:.4f}, b = {fb_ns.extra['b']:.2f}")

**Limb-darkening coefficients for your star:** the values above suit a Sun-like star in the Kepler band.
Look yours up in Claret & Bloemen (2011, Kepler) or Claret (2017, TESS) using the star's $T_\text{eff}$ and
$\log g$ (VizieR tables, linked in the project guide).

## 7. Compare with the literature
How many σ is your result from the published one? Combine both error bars in quadrature:
$n_\sigma = |x - x_\text{pub}| / \sqrt{\sigma^2 + \sigma_\text{pub}^2}$. Within ~2σ = consistent.

In [ ]:
import pandas as pd
rows = [("quick BLS", rp_quick, np.nan), ("trapezoid", ft.rp_rearth, ft.rp_rearth_err)]
if HAVE_BATMAN:
    rows.append(("batman + ρ★ prior", fb.rp_rearth, fb.rp_rearth_err))
table = pd.DataFrame(rows, columns=["method", "Rp [R_earth]", "error"])
table["n_sigma vs published"] = (table["Rp [R_earth]"] - PUBLISHED["rp"]).abs() / np.hypot(table["error"], PUBLISHED["rp_err"])
print(f"published: {PUBLISHED['rp']} ± {PUBLISHED['rp_err']} R_earth")
table.round(3)

### ✏️ Your turn
1. 🟢 Fill in the same table for **your target** (set `STAR`, load it, run §2–§7).
   For TESS 2-min data use `exptime_days=120 / 86400`; for 30-min FFIs `1800 / 86400`.
2. 🟢 Is your radius consistent with the published value? If not, list three possible reasons
   (think: limb darkening, which stellar radius, detrending, **dilution** by nearby stars: TESS pixels are 21″ wide!).
3. 🟡 Rerun the trapezoid fit on the **pass-1** light curve (`flat`). How much does the radius change? In σ?
4. 🔴 Fit with `fit.fit_trapezoid(..., fit_period=True)` to refine the period. Compare its error with the BLS peak width.

<details><summary>💡 Starting point for your target (π Men c, TESS sector 1)</summary>

```python
STAR = {"rstar": 1.10, "rstar_err": 0.02, "mstar": 1.09, "mstar_err": 0.05}
lc = io.get_lightcurve("pi Men", mission="TESS", exptime="120", sector=[1]); lc.meta.update(STAR)
flat = detrend.sigma_clip(detrend.flatten(lc, window_days=1.0))
res = search.bls(flat, min_period=1, max_period=13)
flat2 = detrend.sigma_clip(detrend.flatten_masked(lc, res.period, res.t0, res.duration, window_days=1.0))
rho, rho_err = utils.stellar_density(STAR["rstar"], STAR["mstar"], STAR["rstar_err"], STAR["mstar_err"])
fb = fit.fit_single_transit(flat2, res, model="batman", rho_star=rho, rho_star_err=rho_err,
                            u=(0.28, 0.27), exptime_days=120 / 86400)
print(fb.summary())     # published: 2.04 R_earth (Huang et al. 2018)
```
</details>

In [ ]:
# ✏️ YOUR CODE HERE

### 🚀 Going further (full-timers): MCMC for honest error bars
Covariance errors assume a well-behaved, Gaussian χ² surface. Near the degeneracy it isn't. **MCMC** (Markov
chain Monte Carlo) maps the actual probability distribution of the parameters by random-walking through them.
Install `emcee` and `corner`, then set `RUN_MCMC = True`.
Read: Foreman-Mackey et al. 2013 ([arXiv:1202.3665](https://arxiv.org/abs/1202.3665)) and Hogg &
Foreman-Mackey 2018, *Data analysis recipes: MCMC* ([arXiv:1710.06068](https://arxiv.org/abs/1710.06068)).

In [ ]:
RUN_MCMC = False   # set True after `pip install emcee corner`
if RUN_MCMC and HAVE_BATMAN:
    import emcee, corner
    t, f = flat2.time, flat2.flux
    near = np.abs(((t - fb.t0 + 0.5 * fb.period) % fb.period) - 0.5 * fb.period) < 3 * fb.duration
    t, f = t[near], f[near]
    sig = 1.4826 * np.median(np.abs(f - np.median(f)))

    def log_prob(p):
        t0, k, a, b = p
        if not (0 < k < 0.3 and 1.5 < a < 100 and 0 <= b < 1 + k):
            return -np.inf                       # outside the allowed range: impossible
        m = fit.batman_model(t, t0, fb.period, k, a, b, e["u"], EXPTIME)
        rho_m = 3 * np.pi * a ** 3 / (6.674e-8 * (fb.period * 86400) ** 2)
        return -0.5 * np.sum(((f - m) / sig) ** 2) - 0.5 * ((rho_m - rho) / rho_err) ** 2   # -χ²/2 incl. prior

    ndim, nw = 4, 24
    p0 = np.array([fb.t0, fb.rp_rs, e["a_rs"], min(e["b"], 0.9)]) + 1e-4 * np.random.randn(nw, ndim)
    sampler = emcee.EnsembleSampler(nw, ndim, log_prob)
    sampler.run_mcmc(p0, 3000, progress=True)
    chain = sampler.get_chain(discard=1000, thin=10, flat=True)
    corner.corner(chain, labels=["t0", "Rp/Rs", "a/Rs", "b"]); plt.show()
    k_s = chain[:, 1] * STAR["rstar"] * utils.R_SUN_IN_REARTH
    print(f"Rp = {np.median(k_s):.3f} +{np.percentile(k_s, 84) - np.median(k_s):.3f} "
          f"-{np.median(k_s) - np.percentile(k_s, 16):.3f} R_earth (stellar radius error not included)")

## 🧠 Concept check
**Q1.** The depth is 400 ± 40 ppm and the star radius 1.00 ± 0.07 R☉. Which dominates the radius error?
<details><summary>Answer</summary>Depth: 10 % → k: 5 %. Star: 7 %. The star dominates (total ≈ 8.6 %).</details>

**Q2.** Why does χ² rising by 1 define a 1σ error bar?
<details><summary>Answer</summary>For Gaussian noise, the likelihood ∝ exp(−χ²/2). Moving one parameter by 1σ from the best value raises χ² by exactly 1, so the Δχ² = 1 boundary is the 68 % interval.</details>

**Q3.** Two fits (b = 0 and b = 0.8) are equally good. How do you choose?
<details><summary>Answer</summary>Use outside information: the stellar density implied by each fit vs the catalogue density (a prior), or higher-SNR / shorter-cadence data that resolve the ingress shape.</details>

**Q4.** Your residuals show a "W" shape inside the transit. What is wrong?
<details><summary>Answer</summary>The model shape is wrong, typically missing limb darkening (a flat-bottomed model fitted to a rounded transit) or wrong limb-darkening coefficients.</details>

## 📌 Key takeaways
- Fit on the **masked, second-pass** detrended light curve.
- Fitting = minimising χ²; error bars come from how fast χ² rises (Δχ² = 1).
- Always plot residuals; check error bars with a bootstrap.
- Planet radius error = light-curve error ⊕ star error. Often the star dominates.
- Limb darkening deepens transits; the $b$–$R_p$ degeneracy is broken by a stellar-density prior.
- Quote agreement with the literature in σ.

## 📖 Glossary
**χ²**: sum of squared, error-normalised residuals · **residual**: data − model · **covariance matrix**: error bars
(and correlations) from the χ² curvature · **bootstrap**: errors from refitting resampled data · **limb darkening**: a star's
dimmer edge · **impact parameter b**: how far from centre the planet crosses · **prior**: outside information
added to a fit · **supersampling**: averaging the model over each exposure.

### ✅ Week 3 Checkpoint
- [ ] Kepler-10 b radius with error bar, compared to the published 1.47 R⊕ (in σ)
- [ ] Your target's radius with error bar + the comparison table + residual plot
- [ ] Draft of the **Methods** section of your report (data → detrend → BLS → fit), ~1 page

**➡️ Next:** Notebook 04, the twist. How many planets like this would your pipeline have *missed*?